# Guided Data Cleaning Projects
## Economics, Business, Data Science, and Supply Chain

This notebook contains **4 guided Data Cleaning practice projects**:

1. **Retail Sales** — Business Analytics  
2. **Household Economic Survey** — Economics  
3. **Customer Churn** — Data Science / CRM  
4. **Logistics Delivery** — Operations / Supply Chain  

---

### Course & Contact
- **Course:** Data Analysis with Python (DSAI1005)
- **Lecturer:** Dr. Minh Duc Vu (`minhvd@neu.edu.vn`)
- **Department:** School of Data Science and Artificial Intelligence – National Economics University (NEU)

---

### Recommended Workflow for Each Project
1. **Initial Quality Assessment:** `shape`, `info()`, `describe()`, missing values, duplicates, dtypes.
2. **Structural Cleaning:** column names, duplicate rows/keys, string normalization.
3. **Data Type & Date Harmonization:** numeric conversion, date parsing (`format="mixed"`).
4. **Missing Values & Imputation:** missing mechanism analysis, median/mode/grouped imputation.
5. **Business Rules & Valid Ranges:** identify impossible or contradictory domain values.
6. **Outlier Detection:** Boxplots, IQR thresholds, domain-aware handling.
7. **Validation & Verification:** post-cleaning assertions, diagnostic summary reports.

## 0. Environment Setup

Key libraries:

- `pandas`: Tabular data manipulation;
- `numpy`: Numerical operations and handling `NaN`;
- `matplotlib` / `seaborn`: Exploratory data visualization;
- `pathlib`: File path management.

This notebook assumes that the data files reside in the same directory as the notebook or are uploaded to your working runtime:

```text
retail_sales_dirty.csv
household_economic_survey_dirty.csv
customer_churn_dirty.csv
logistics_delivery_dirty.csv
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

print("Libraries loaded successfully.")

### Data Loading Guide for Google Colab

If the dataset files are not present in your Colab environment, upload them manually:

```python
from google.colab import files
uploaded = files.upload()
```

Then read the dataset:

```python
df = pd.read_csv("file_name.csv")
```

When running locally, simply place the CSV files in the same directory as this notebook.

# Project 1 — Retail Sales Data Cleaning

## Context

A retail store chain needs to analyze:

- Revenue by geographic region;
- Top-selling product lines;
- Preferred customer payment methods;
- General purchasing behavior.

The raw transactional dataset contains common real-world flaws:

- Duplicate `Order_ID` entries;
- Inconsistent `Region` and `Payment_Method` labels;
- Heterogeneous date formatting;
- Corrupted percentage discounts;
- Missing prices;
- Negative quantities;
- Revenue amounts that contradict unit price and quantity;
- Potential extreme outliers.

In [ ]:
# Load dataset
retail = pd.read_csv("retail_sales_dirty.csv")

retail.head()

## Task 1 — Data Quality Assessment

### Theory Recap

Before performing any cleaning, systematically assess:

```python
df.shape
df.info()
df.describe()
df.isna().sum()
df.duplicated().sum()
```

To categorize numerical vs. categorical columns:

```python
df.select_dtypes(include=np.number)
df.select_dtypes(include=["object", "category"])
```

Identifiers (e.g., `Order_ID`, `Customer_ID`) should not be treated as regular numerical features.

In [ ]:
# TODO 1.1 — Check dimensions
print("Shape:", ________)

# TODO 1.2 — Inspect schema and data types
# retail.________()

# TODO 1.3 — Descriptive statistics
# display(retail.________())

# TODO 1.4 — Missing value counts
# display(retail.________().sum())

# TODO 1.5 — Duplicate rows count
# print("Duplicate rows:", retail.________().sum())

In [ ]:
# TODO 1.6 — Identify variable types

numeric_cols = retail.select_dtypes(
    include=________
).columns.tolist()

categorical_cols = retail.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical:", numeric_cols)
print("Categorical:", categorical_cols)

## Task 2 — Duplicate Records

### Theory Recap

- `df.duplicated()` identifies completely identical rows across all columns.
- `df.duplicated(subset=[...])` checks for duplicates against business entity keys.
- `keep=False` displays **all** occurrences involved in duplicate records.

Example:

```python
df[df.duplicated(subset=["Order_ID"], keep=False)]
```

In [ ]:
# TODO 2.1 — Display duplicated Order_IDs

duplicate_orders = retail[
    retail.________(
        subset=["Order_ID"],
        keep=False
    )
]

duplicate_orders.head(20)

In [ ]:
# TODO 2.2 — Count duplicated Order_IDs

duplicate_count = retail.________(
    subset=["Order_ID"]
).sum()

print("Duplicate Order_ID count:", duplicate_count)

### Question for Reflection

If two rows share the same:

```text
Customer_ID
Product
Revenue
```

but possess different `Order_ID` values, are they necessarily duplicate records?

> Provide your answer in Markdown before deciding whether to apply `drop_duplicates()`.

## Task 3 — Standardizing Categorical Variables

### Theory Recap

Helpful methods:

```python
.str.strip()
.str.lower()
.str.title()
.replace()
value_counts(dropna=False)
```

Standard workflow:

```text
inspect unique values
→ strip whitespace
→ normalize case (lower/title)
→ map synonymous categories
→ re-verify distribution
```

In [ ]:
# Inspect current categorical distributions
print(retail["Region"].value_counts(dropna=False))
print()
print(retail["Payment_Method"].value_counts(dropna=False))

In [ ]:
# TODO 3.1 — Standardize Region

retail["Region"] = (
    retail["Region"]
    .str.________()
    .str.________()
)

# Suggested mapping:
region_map = {
    "ha noi": "hanoi",
    "hcm": "ho chi minh",
    "danang": "da nang"
}

retail["Region"] = retail["Region"].replace(
    ________
)

retail["Region"].value_counts(dropna=False)

In [ ]:
# TODO 3.2 — Standardize Payment_Method

retail["Payment_Method"] = (
    retail["Payment_Method"]
    .str.________()
    .str.________()
)

payment_map = {
    "credit-card": "credit card",
    "ewallet": "e-wallet"
}

retail["Payment_Method"] = retail["Payment_Method"].replace(
    ________
)

retail["Payment_Method"].value_counts(dropna=False)

## Task 4 — Standardizing Date Formats

### Theory Recap

Parse dates with:

```python
pd.to_datetime(
    series,
    errors="coerce",
    format="mixed"
)
```

`errors="coerce"` converts unparseable date strings into `NaT`.

Always verify unparseable values afterward:

```python
df["Date"].isna().sum()
```

In [ ]:
# TODO 4.1 — Convert Order_Date to datetime

retail["Order_Date"] = pd.to_datetime(
    retail["Order_Date"],
    format="mixed",
    dayfirst=True,
    errors="________"
)

print("Invalid dates count:", retail["Order_Date"].isna().sum())

## Task 5 — Missing Values

### Theory Recap

Quantify missingness:

```python
df.isna().sum()
df.isna().mean() * 100
```

Impute numerical features globally:

```python
series.fillna(series.median())
```

Or conditionally by category group:

```python
df.groupby("Product")["Unit_Price"].transform("median")
```

In [ ]:
# TODO 5.1 — Generate missing values summary table

missing_summary = pd.DataFrame({
    "missing_count": retail.________().sum(),
    "missing_percent": (retail.________().mean() * 100).round(2)
})

missing_summary.sort_values(
    "missing_percent",
    ascending=False
)

In [ ]:
# TODO 5.2 — Convert Discount to numeric scale [0, 1]

# Guidance:
# 1. astype(str)
# 2. str.replace("%", "")
# 3. pd.to_numeric(errors="coerce")
# 4. if value > 1, divide by 100

discount = (
    retail["Discount"]
    .astype(str)
    .str.replace("%", "", regex=False)
)

retail["Discount"] = pd.to_numeric(
    discount,
    errors="coerce"
)

retail.loc[
    retail["Discount"] > 1,
    "Discount"
] = retail.loc[
    retail["Discount"] > 1,
    "Discount"
] / 100

retail["Discount"].describe()

In [ ]:
# TODO 5.3 — Impute Unit_Price by Product median

median_by_product = (
    retail
    .groupby("Product")["Unit_Price"]
    .transform("________")
)

retail["Unit_Price"] = retail["Unit_Price"].fillna(
    ________
)

## Task 6 — Business Rules Validation

### Theory Recap

A business rule defines expected domain relationships between features:

\[
Expected\ Revenue = Quantity 	imes Unit\_Price 	imes (1 - Discount)
\]

Domain constraints:

```text
Quantity > 0
Unit_Price > 0
0 <= Discount <= 1
```

In [ ]:
# TODO 6.1 — Identify invalid Quantity values

invalid_quantity = retail[
    retail["Quantity"] ________ 0
]

invalid_quantity

In [ ]:
# TODO 6.2 — Identify invalid Unit_Price values

invalid_price = retail[
    retail["Unit_Price"] ________ 0
]

invalid_price

In [ ]:
# TODO 6.3 — Identify Discount outside [0, 1]

invalid_discount = retail[
    ~retail["Discount"].________(0, 1)
]

invalid_discount

In [ ]:
# TODO 6.4 — Compute Expected Revenue and detect discrepancies

retail["Expected_Revenue"] = (
    retail["________"]
    * retail["________"]
    * (1 - retail["________"])
)

retail["Revenue_Diff"] = (
    retail["Revenue"]
    - retail["Expected_Revenue"]
)

revenue_mismatch = retail[
    retail["Revenue_Diff"].abs() > 1
]

revenue_mismatch.head()

## Task 7 — Outlier Detection

### Theory Recap

Boxplots:

```python
sns.boxplot(x=df["Revenue"])
```

Interquartile Range (IQR):

```python
q1 = s.quantile(0.25)
q3 = s.quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
```

Do not automatically drop outliers. Categorize them:

```text
- Data Entry Error (fix or discard)
- Valid High-Value Transaction (retain)
- Anomaly / Fraud Candidate (flag)
```

In [ ]:
# TODO 7.1 — Plot Boxplot for Revenue

plt.figure(figsize=(8, 3))
sns.________(
    x=retail["Revenue"]
)
plt.title("Revenue Boxplot")
plt.show()

In [ ]:
# TODO 7.2 — Write reusable IQR outlier detection function

def iqr_outliers(series):
    q1 = series.quantile(________)
    q3 = series.quantile(________)
    iqr = ________ - ________
    lower = q1 - 1.5 * ________
    upper = q3 + 1.5 * ________

    mask = (
        (series < lower)
        |
        (series > upper)
    )
    return mask

outlier_mask = iqr_outliers(retail["Revenue"])
print("Number of Revenue outliers:", outlier_mask.sum())
retail[outlier_mask].head()

## Task 8 — Validation Report

### Guidance

Use programmatic assertions:

```python
assert condition
```

or build a diagnostic dictionary summary:

```python
checks = {
    "duplicate_order_id": ...,
    "missing_values": ...,
    "invalid_quantity": ...
}
```

In [ ]:
# TODO 8 — Generate validation report

retail_validation = {
    "duplicate_order_id": retail.duplicated(
        subset=["Order_ID"]
    ).sum(),

    "missing_total": retail.________().sum().sum(),

    "invalid_quantity": (
        retail["Quantity"] <= 0
    ).sum(),

    "invalid_price": (
        retail["Unit_Price"] <= 0
    ).sum(),

    "revenue_mismatches": (
        retail["Revenue_Diff"].abs() > 1
    ).sum()
}

retail_validation

## Post-Cleaning Analysis

Answer the following business questions using your cleaned dataset:

1. Which region generates the highest total revenue?
2. Which product line sells the highest volume (total quantity)?
3. What is the Average Order Value (AOV)?
4. Which payment method is most frequently used?
5. How significantly do metrics differ between the raw and cleaned datasets?

In [ ]:
# TODO — Post-cleaning exploratory analysis

# 1. Revenue by Region
# retail.groupby("Region")["Revenue"].sum().sort_values(ascending=False)

# 2. Quantity by Product
# ...

# 3. Average Order Value
# ...

# 4. Payment Method counts
# ...

# Project 2 — Household Economic Survey Cleaning

## Context

An economics research team is analyzing a household living standards dataset to study:

- Household income and consumption patterns;
- Household size and demographic composition;
- Employment status and educational attainment;
- Regional economic disparities (Urban vs. Rural).

The raw survey data contains typical survey artifacts:

- Sentinel missing codes (`-1`, `999`, `"Unknown"`, `"N/A"`);
- Mixed monetary units (`VND` vs. `thousand VND`);
- Impossible household sizes (`Household_Size <= 0`);
- Categorical inconsistencies;
- Economic business rule violations ($Food + Housing > Total\ Consumption$);
- Highly skewed income distribution with extreme values.

In [ ]:
household = pd.read_csv(
    "household_economic_survey_dirty.csv"
)

household.head()

## Task 1 — Standardizing Sentinel Missing Codes

### Theory Recap

Missing values in surveys are often encoded as sentinel numeric or string codes.

We replace them with true `NaN`:

```python
df.replace(
    ["Unknown", "N/A", "NULL", -1, 999],
    np.nan
)
```

Only convert values that genuinely represent missingness or refusal to answer.

In [ ]:
# TODO 1 — Standardize sentinel missing codes

missing_codes = [
    "Unknown",
    "N/A",
    "NULL",
    ________
]

household = household.replace(
    missing_codes,
    np.nan
)

## Task 2 — Missing Value Analysis

### Guidance

Quantify missingness overall and across demographic strata:

```python
df.isna().sum()
df.isna().mean() * 100
```

Evaluate whether missingness depends on employment or education:

```python
df.groupby("Employment_Status")["Monthly_Income"].apply(
    lambda s: s.isna().mean()
)
```

This helps distinguish between MCAR (Missing Completely at Random), MAR, and MNAR mechanisms.

In [ ]:
# TODO 2.1 — Missing summary

household_missing = pd.DataFrame({
    "count": household.________().sum(),
    "percent": (
        household.________().mean() * 100
    ).round(2)
})

household_missing

In [ ]:
# TODO 2.2 — Missing Income rate by Employment_Status

missing_income_by_employment = (
    household
    .groupby("Employment_Status")["Monthly_Income"]
    .apply(
        lambda s: s.________().mean() * 100
    )
)

missing_income_by_employment

## Task 3 — Harmonizing Monetary Units

### Theory Recap

The dataset may mix different units of denomination:

```text
VND
thousand VND
```

When:

```text
Monetary_Unit == "thousand VND"
```

the monetary figures must be scaled by `1000`.

Use `.loc[mask, columns]` to update values safely.

In [ ]:
money_cols = [
    "Monthly_Income",
    "Monthly_Consumption",
    "Food_Expenditure",
    "Housing_Expenditure"
]

# Ensure numeric data types first
for col in money_cols:
    household[col] = pd.to_numeric(
        household[col],
        errors="coerce"
    )

In [ ]:
# TODO 3 — Convert thousand VND -> VND

mask_thousand = (
    household["Monetary_Unit"] == "________"
)

household.loc[
    mask_thousand,
    money_cols
] = (
    household.loc[
        mask_thousand,
        money_cols
    ] * ________
)

household.loc[
    mask_thousand,
    "Monetary_Unit"
] = "VND"

household[money_cols].describe()

## Task 4 — Valid Ranges and Economic Consistency Rules

Economic consistency constraints:

```text
Household_Size >= 1
Income >= 0
Consumption >= 0
Food >= 0
Housing >= 0

Food_Expenditure + Housing_Expenditure <= Monthly_Consumption
```

In [ ]:
# TODO 4.1 — Detect invalid Household_Size

invalid_household_size = household[
    household["Household_Size"] ________ 1
]

invalid_household_size

In [ ]:
# TODO 4.2 — Validate expenditure component logic

household["Core_Expenses"] = (
    household["Food_Expenditure"]
    + household["Housing_Expenditure"]
)

expense_violation = household[
    household["Core_Expenses"]
    ________
    household["Monthly_Consumption"]
]

expense_violation.head()

## Task 5 — Outlier Analysis and Skewness

### Guidance

Analyze heavy-tailed distributions using:

```python
sns.boxplot()
IQR method
np.log1p()
```

`np.log1p(x)` computes $\ln(1 + x)$, which compresses extreme positive skewness while safely handling zero incomes.

In [ ]:
# TODO 5.1 — Boxplot of Monthly Income

plt.figure(figsize=(8, 3))
sns.________(
    x=household["Monthly_Income"]
)
plt.title("Monthly Income Distribution")
plt.show()

In [ ]:
# TODO 5.2 — Logarithmic transformation

household["Income_Log"] = np.________(
    household["Monthly_Income"]
)

household[["Monthly_Income", "Income_Log"]].head()

## Task 6 — Imputation Strategies

Compare different imputation approaches:

```text
1. Global Median
2. Median grouped by Region
3. Median grouped by Employment_Status & Education
```

Implementation with `transform`:

```python
df.groupby("Region")["Monthly_Income"].transform("median")
```

In [ ]:
# TODO 6.1 — Global median income

income_median = household["Monthly_Income"].________()

income_median

In [ ]:
# TODO 6.2 — Impute by Regional Median

median_income_region = (
    household
    .groupby("Region")["Monthly_Income"]
    .transform("________")
)

household["Income_Imputed_By_Region"] = (
    household["Monthly_Income"]
    .fillna(________)
)

## Task 7 — Comprehensive Validation Function

Build a function returning a diagnostic dictionary:

```python
{
    "missing_income": ...,
    "invalid_household_size": ...,
    "negative_consumption": ...,
    "expense_rule_violation": ...,
    "duplicate_household_id": ...
}
```

In [ ]:
# TODO 7 — Complete household validation function

def validate_household_data(df):
    return {
        "missing_income": df["Monthly_Income"].________().sum(),

        "invalid_household_size": (
            df["Household_Size"] < 1
        ).sum(),

        "negative_consumption": (
            df["Monthly_Consumption"] < 0
        ).sum(),

        "expense_rule_violation": (
            (df["Food_Expenditure"] + df["Housing_Expenditure"])
            > df["Monthly_Consumption"]
        ).sum(),

        "duplicate_household_id": df.duplicated(
            subset=["Household_ID"]
        ).sum()
    }

validate_household_data(household)

## Post-Cleaning Economic Analysis

Investigate the following research questions:

1. What is the Median Income across geographical regions?
2. How does Average Consumption differ between Urban and Rural households?
3. What is the average Household Consumption-to-Income ratio?
4. Which educational attainment group reports the highest median income?
5. How does median imputation affect the estimated mean income?

In [ ]:
# TODO — Economic analysis examples

# Median Income by Region
# household.groupby("Region")["Monthly_Income"].median()

# Urban vs Rural consumption
# household.groupby("Urban_Rural")["Monthly_Consumption"].mean()

# Consumption-to-Income ratio
# household["Consumption_Income_Ratio"] = (
#     household["Monthly_Consumption"] / household["Monthly_Income"]
# )

# Project 3 — Customer Churn CRM Cleaning

## Context

Customer Relationship Management (CRM) data merged from disparate transactional and support databases exhibits typical integration issues:

- Duplicate `Customer_ID` records;
- Inconsistent categorical strings (`Gender`, `Region`, `Contract_Type`);
- Non-standardized target variable `Churn` (`"Yes"`, `"Y"`, `"1"`, `"No"`, `"N"`, `"0"`);
- Numbers stored as strings with commas (`"1,250.50"`);
- Invalid `Age` values;
- Missing `Monthly_Fee` entries;
- Inverted timestamps (`Join_Date > Last_Login`);
- Potential target leakage features.

In [ ]:
churn = pd.read_csv(
    "customer_churn_dirty.csv"
)

churn.head()

## Task 1 — Schema and Data Integrity Check

Guidance:

```python
df.info()
df.dtypes
df.nunique()
df.isna().sum()
df.duplicated(subset=["Customer_ID"]).sum()
```

In [ ]:
# TODO 1 — Schema examination

# churn.________()
# display(churn.dtypes)
# display(churn.________())
# display(churn.isna().sum())

print(
    "Duplicate Customer_ID count:",
    churn.duplicated(
        subset=["Customer_ID"]
    ).sum()
)

## Task 2 — Standardizing Target Variable `Churn`

Goal: Map all target variants into binary numerical indicators:

```text
No, N, 0   -> 0 (Retained)
Yes, Y, 1  -> 1 (Churned)
```

Useful pattern:

```python
.astype(str)
.str.strip()
.str.lower()
.replace(mapping)
pd.to_numeric(errors="coerce")
```

In [ ]:
# TODO 2 — Standardize and map Churn target

churn_target = (
    churn["Churn"]
    .astype(str)
    .str.________()
    .str.________()
)

churn_map = {
    "yes": 1,
    "y": 1,
    "1": 1,
    "no": 0,
    "n": 0,
    "0": 0
}

churn["Churn_Clean"] = churn_target.________(
    churn_map
)

churn["Churn_Clean"].value_counts(dropna=False)

## Task 3 — Standardizing Categorical Variables

Key features:

```text
Gender
Region
Contract_Type
Payment_Method
```

Workflow:

```text
strip whitespace
→ lowercase
→ map synonyms
→ inspect value_counts()
```

In [ ]:
# TODO 3.1 — Standardize Gender

gender_clean = (
    churn["Gender"]
    .astype(str)
    .str.________()
    .str.________()
)

gender_map = {
    "m": "male",
    "male": "male",
    "f": "female",
    "female": "female"
}

churn["Gender"] = gender_clean.replace(
    ________
)

churn["Gender"].value_counts(dropna=False)

## Task 4 — Numeric Parsing and Cleaning

Convert strings with thousands separators to numbers:

```python
pd.to_numeric(
    df["column"].astype(str).str.replace(",", ""),
    errors="coerce"
)
```

In [ ]:
# TODO 4 — Clean numeric features

churn["Total_Spending"] = pd.to_numeric(
    churn["Total_Spending"]
    .astype(str)
    .str.replace(",", "", regex=False),
    errors="________"
)

churn["Monthly_Fee"] = pd.to_numeric(
    churn["Monthly_Fee"],
    errors="coerce"
)

churn["Support_Tickets"] = pd.to_numeric(
    churn["Support_Tickets"],
    errors="coerce"
)

churn[["Total_Spending", "Monthly_Fee", "Support_Tickets"]].dtypes

## Task 5 — Date Parsing and Feature Engineering

Parse timestamps with:

```python
pd.to_datetime(..., format="mixed", errors="coerce")
```

Engineer temporal features:

```text
Customer_Tenure_Days
Days_Since_Last_Login
```

Relative to a reference observation date:

```python
reference_date = pd.Timestamp("2026-09-01")
```

In [ ]:
# TODO 5.1 — Parse timestamp columns

for col in ["Join_Date", "Last_Login"]:
    churn[col] = pd.to_datetime(
        churn[col],
        format="mixed",
        dayfirst=True,
        errors="________"
    )

In [ ]:
# TODO 5.2 — Engineer temporal tenure features

reference_date = pd.Timestamp("2026-09-01")

churn["Customer_Tenure_Days"] = (
    reference_date
    - churn["________"]
).dt.days

churn["Days_Since_Last_Login"] = (
    reference_date
    - churn["________"]
).dt.days

churn[["Customer_Tenure_Days", "Days_Since_Last_Login"]].describe()

## Task 6 — Business Logic Constraints

Domain validity rules:

```text
18 <= Age <= 100
Monthly_Fee >= 0
Total_Spending >= 0
Support_Tickets >= 0
Join_Date <= Last_Login
```

In [ ]:
# TODO 6.1 — Detect invalid Age values

invalid_age = churn[
    ~churn["Age"].________(18, 100)
]

invalid_age.head()

In [ ]:
# TODO 6.2 — Verify chronological order

invalid_date_order = churn[
    churn["Join_Date"]
    ________
    churn["Last_Login"]
]

invalid_date_order.head()

## Task 7 — Missing Value Treatment

When preparing data for machine learning models:

> **Data Leakage Precaution:** Imputers must be fit exclusively on the training split (`X_train`), never on the combined dataset.

In this exploratory cleaning stage, quantify missingness and formulate appropriate imputation strategies.

In [ ]:
# TODO 7 — Quantify missing value rates

churn_missing = (
    churn.________().mean() * 100
).sort_values(
    ascending=False
)

churn_missing

## Task 8 — Outlier Detection

Examine numerical distributions:

```text
Total_Spending
Monthly_Fee
Support_Tickets
```

Use boxplots and the IQR method. Distinguish between:

```text
- High-value VIP customer (legitimate extreme)
- Highly dissatisfied customer (excessive tickets)
- Corrupted data entry
```

In [ ]:
# TODO 8 — Outlier boxplots

cols = [
    "Total_Spending",
    "Monthly_Fee",
    "Support_Tickets"
]

for col in cols:
    plt.figure(figsize=(8, 2.5))
    sns.________(
        x=churn[col]
    )
    plt.title(col)
    plt.show()

## Task 9 — Target Leakage Audit

### Theory Recap

**Target Leakage** occurs when a predictor incorporates information that would not be available at inference time (before churn occurs).

Suspect variables:

```text
Closure_Date
Cancellation_Reason
Account_Status (if set to "Closed")
```

Such features must be removed from predictive modeling pipelines.

In [ ]:
# TODO 9 — Identify leakage features to exclude from modeling

leakage_features = [
    "________",
    "________"
]

print("Potential target leakage features:", leakage_features)

## Post-Cleaning CRM Analytics

Investigate the following customer behavior questions:

1. What is the baseline churn rate across the cleaned customer base?
2. Does churn rate vary significantly by `Contract_Type` (e.g., Month-to-Month vs. Annual)?
3. Is higher `Support_Tickets` associated with increased churn probability?
4. How does `Tenure` relate to customer retention?

In [ ]:
# TODO — Churn analysis examples

# 1. Overall Churn Rate
# churn["Churn_Clean"].mean()

# 2. Churn by Contract Type
# churn.groupby("Contract_Type")["Churn_Clean"].mean()

# 3. Churn by Support Tickets
# churn.groupby("Support_Tickets")["Churn_Clean"].mean()

# Project 4 — Logistics and Delivery Data Cleaning

## Context

A third-party logistics (3PL) carrier manages package shipments across domestic and regional hubs.

The analytics team is examining:

- On-time delivery performance and delivery delays;
- Shipping cost efficiency per kilometer;
- Carrier performance metrics.

The operational logs contain common data hygiene challenges:

- Duplicate shipment tracking numbers (`Shipment_ID`);
- Inconsistent `Carrier` and `Status` naming;
- Chronological date anomalies (`Ship_Date > Actual_Delivery_Date`);
- Mixed measurement units for distance (`miles` vs. `km`) and package weight (`lb`, `g`, `kg`);
- Missing actual delivery timestamps on delivered parcels;
- Business rule inconsistencies ($Cost < 0$, $Weight \le 0$).

In [ ]:
logistics = pd.read_csv(
    "logistics_delivery_dirty.csv"
)

logistics.head()

## Task 1 — Detecting Duplicate Shipments

Inspect tracking identifiers:

```python
df[df.duplicated(subset=["Shipment_ID"], keep=False)]
```

In [ ]:
# TODO 1 — Display duplicated Shipment_IDs

dup_shipments = logistics[
    logistics.________(
        subset=["Shipment_ID"],
        keep=False
    )
]

dup_shipments.head(20)

## Task 2 — Standardizing Carrier and Status

Clean text and map synonyms:

```python
.str.strip()
.str.lower()
.replace(mapping)
```

In [ ]:
# TODO 2.1 — Standardize Carrier

logistics["Carrier"] = (
    logistics["Carrier"]
    .str.________()
    .str.________()
)

carrier_map = {
    "dhl express": "dhl",
    "fed ex": "fedex",
    "vietnam post": "vnpost",
    "giao hang nhanh": "ghn"
}

logistics["Carrier"] = logistics["Carrier"].replace(
    ________
)

logistics["Carrier"].value_counts(dropna=False)

In [ ]:
# TODO 2.2 — Standardize Status

logistics["Status"] = (
    logistics["Status"]
    .str.strip()
    .str.lower()
)

status_map = {
    "late": "delayed",
    "canceled": "cancelled",
    "in-transit": "in transit",
    "intransit": "in transit"
}

logistics["Status"] = logistics["Status"].replace(
    ________
)

logistics["Status"].value_counts(dropna=False)

## Task 3 — Parsing and Validating Dates

Parse columns:

```text
Ship_Date
Expected_Delivery_Date
Actual_Delivery_Date
```

Verify chronological consistency:

```text
Ship_Date <= Expected_Delivery_Date
Ship_Date <= Actual_Delivery_Date
```

In [ ]:
# TODO 3.1 — Parse logistics dates

date_cols = [
    "Ship_Date",
    "Expected_Delivery_Date",
    "Actual_Delivery_Date"
]

for col in date_cols:
    logistics[col] = pd.to_datetime(
        logistics[col],
        format="mixed",
        dayfirst=True,
        errors="________"
    )

In [ ]:
# TODO 3.2 — Check chronological date consistency

invalid_expected = logistics[
    logistics["Ship_Date"]
    ________
    logistics["Expected_Delivery_Date"]
]

invalid_actual = logistics[
    logistics["Ship_Date"]
    ________
    logistics["Actual_Delivery_Date"]
]

print("Invalid expected delivery dates count:", len(invalid_expected))
print("Invalid actual delivery dates count  :", len(invalid_actual))

## Task 4 — Harmonizing Measurement Units

### Distance
\[
1\ 	ext{mile} pprox 1.60934\ 	ext{km}
\]

### Weight
\[
1\ 	ext{lb} pprox 0.453592\ 	ext{kg}
\]
\[
1000\ 	ext{g} = 1\ 	ext{kg}
\]

In [ ]:
# TODO 4.1 — Standardize Distance to kilometers (km)

mask_miles = logistics["Distance_Unit"].str.lower() == "miles"

logistics.loc[
    mask_miles,
    "Distance"
] = (
    logistics.loc[
        mask_miles,
        "Distance"
    ] * ________
)

logistics.loc[
    mask_miles,
    "Distance_Unit"
] = "km"

logistics["Distance"].describe()

In [ ]:
# TODO 4.2 — Standardize Weight to kilograms (kg)

mask_g = logistics["Weight_Unit"].str.lower() == "g"
mask_lb = logistics["Weight_Unit"].str.lower() == "lb"

logistics.loc[
    mask_g,
    "Weight"
] = (
    logistics.loc[
        mask_g,
        "Weight"
    ] / ________
)

logistics.loc[
    mask_lb,
    "Weight"
] = (
    logistics.loc[
        mask_lb,
        "Weight"
    ] * ________
)

logistics.loc[
    mask_g | mask_lb,
    "Weight_Unit"
] = "kg"

logistics["Weight"].describe()

## Task 5 — Context-Aware Missing Value Analysis

### Crucial Domain Distinction

A missing `Actual_Delivery_Date` is **not necessarily an error**:

```text
Status = "in transit"  -> Delivery date is genuinely not yet known (Valid NaN)
Status = "cancelled"   -> Delivery date will never occur (Valid NaN)
Status = "delivered"   -> Delivery date MUST exist (Business Rule Violation!)
```

In [ ]:
# TODO 5 — Identify Delivered shipments missing Actual_Delivery_Date

missing_delivered_date = logistics[
    (logistics["Status"] == "delivered")
    &
    logistics["Actual_Delivery_Date"].________()
]

print("Delivered shipments missing delivery date:", len(missing_delivered_date))
missing_delivered_date.head()

## Task 6 — Feature Engineering: Delivery Delay

Calculate delivery delay in days:

\[
Delivery\ Delay = Actual\ Delivery\ Date - Expected\ Delivery\ Date
\]

In Pandas:

```python
(df["Actual_Delivery_Date"] - df["Expected_Delivery_Date"]).dt.days
```

- Positive value ($> 0$): Delivery was late.
- Zero ($= 0$): On-time delivery.
- Negative value ($< 0$): Early delivery.

In [ ]:
# TODO 6 — Compute Delivery Delay in days

logistics["Delivery_Delay"] = (
    logistics["________"]
    - logistics["________"]
).dt.days

logistics["Delivery_Delay"].describe()

## Task 7 — Outlier Analysis

Inspect continuous metrics:

```text
Distance
Weight
Shipping_Cost
Delivery_Delay
```

Employ boxplots and IQR thresholds to flag anomalous values.

In [ ]:
# TODO 7 — Outlier boxplots

for col in [
    "Distance",
    "Weight",
    "Shipping_Cost",
    "Delivery_Delay"
]:
    plt.figure(figsize=(8, 2.5))
    sns.________(
        x=logistics[col]
    )
    plt.title(col)
    plt.show()

## Task 8 — Business Rule Audit

Validate core physical and financial constraints:

```text
Distance > 0
Weight > 0
Shipping_Cost >= 0

Status == "delivered" -> Actual_Delivery_Date must not be null
```

In [ ]:
# TODO 8 — Count business-rule violations

business_rules = {
    "invalid_distance": (
        logistics["Distance"] <= 0
    ).sum(),

    "invalid_weight": (
        logistics["Weight"] <= 0
    ).sum(),

    "negative_cost": (
        logistics["Shipping_Cost"] < 0
    ).sum(),

    "missing_delivered_date": (
        (logistics["Status"] == "delivered")
        & logistics["Actual_Delivery_Date"].isna()
    ).sum()
}

business_rules

## Task 9 — Logistics Validation Suite

Build an automated diagnostic function checking:

```text
duplicate_shipments
invalid_dates
missing_delivered_dates
negative_costs
invalid_weight
invalid_distance
```

In [ ]:
# TODO 9 — Complete logistics validation function

def validate_logistics_data(df):
    return {
        "duplicate_shipments": df.duplicated(
            subset=["Shipment_ID"]
        ).sum(),

        "invalid_dates": (
            df["Actual_Delivery_Date"]
            <
            df["Ship_Date"]
        ).sum(),

        "missing_delivered_dates": (
            (df["Status"] == "delivered")
            & df["Actual_Delivery_Date"].isna()
        ).sum(),

        "negative_costs": (
            df["Shipping_Cost"] < 0
        ).sum(),

        "invalid_weight": (
            df["Weight"] <= 0
        ).sum(),

        "invalid_distance": (
            df["Distance"] <= 0
        ).sum()
    }

validate_logistics_data(logistics)

## Post-Cleaning Supply Chain Analysis

Investigate the following operational questions:

1. Which carrier exhibits the lowest average delivery delay?
2. Which carrier offers the most competitive shipping cost per kilometer?
3. Which Origin–Destination shipping corridor suffers from the greatest delays?
4. What is the overall distribution of shipment statuses (`Delivered`, `Delayed`, `Cancelled`)?
5. Are shipping cost outliers explained by disproportionate package weight or distance?

In [ ]:
# TODO — Logistics operational analysis

# 1. Average delay by carrier
# logistics.groupby("Carrier")["Delivery_Delay"].mean()

# 2. Shipping cost per kilometer
# logistics["Cost_per_km"] = logistics["Shipping_Cost"] / logistics["Distance"]

# 3. Delay by shipping corridor
# logistics.groupby(["Origin", "Destination"])["Delivery_Delay"].mean()

# General Requirements Across All 4 Projects

## Phase 1 — Data Quality Assessment

For each project, produce a comprehensive data quality diagnostic table:

| Variable | Type | Missing Count | Missing % | Unique Values | Suspected Data Quality Issues |
|---|---|---:|---:|---:|---|

Guidance:

```python
quality = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2),
    "unique": df.nunique()
})
```

In [ ]:
# TODO — Write a reusable data quality reporting function

def data_quality_report(df):
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "missing": df.________().sum(),
        "missing_pct": (
            df.________().mean() * 100
        ).round(2),
        "unique": df.________()
    })

# Example test:
# data_quality_report(retail)

## Phase 2 — Cleaning Audit Log

Every data cleaning modification must be documented systematically in an audit log.

Example:

```python
cleaning_log = []

cleaning_log.append({
    "step": 1,
    "variable": "Region",
    "problem": "Inconsistent casing and spelling variants",
    "action": "Lowercase transformation + dictionary mapping",
    "reason": "Harmonize synonymous geographical entities"
})
```

In [ ]:
# TODO — Initialize and populate your cleaning audit log

cleaning_log = []

# Example:
# cleaning_log.append({
#     "step": 1,
#     "variable": "Region",
#     "problem": "...",
#     "action": "...",
#     "reason": "..."
# })

pd.DataFrame(cleaning_log)

## Phase 3 — Before vs. After Validation Audit

Systematically compare dataset metrics before and after cleaning:

```text
- Total Row Count
- Total Column Count
- Duplicate Key Count
- Total Missing Value Count
- Out-of-Range Value Count
- Statistical Outlier Count
- Business Rule Violations Count
```

> **Evaluation Philosophy:** A data cleaning project is not judged merely by whether the final dataset is free of `NaN`s or outliers.  
> A successful project reflects:
> 1. Transparent rationale for every transformation;
> 2. Respect for domain constraints and business rules;
> 3. Reproducible, well-structured, modular Python code.

# Suggested Evaluation Rubric

| Component | Weight | Key Assessment Criteria |
|---|---:|---|
| **Data Quality Assessment** | 15% | Comprehensive inspection of shape, types, missing values, duplicates, and initial findings |
| **Structural Cleaning & Deduplication** | 15% | Accurate deduplication by business keys, snake_case column names, string normalization |
| **Missing-Value Strategy** | 15% | Justified imputation (mean/median/mode/grouped) without introducing artificial bias |
| **Outlier & Unit Harmonization** | 15% | Standardizing physical/monetary units, IQR/Boxplot detection, avoiding blind deletion |
| **Domain & Business Rule Validation** | 15% | Rigorous verification of domain constraints, mathematical consistency, and assertions |
| **Code Quality & Reproducibility** | 10% | Clean, vectorized Pandas code, modular functions, meaningful variable naming |
| **Analytical Interpretation & Audit Log** | 15% | Documented cleaning log, before-vs-after comparison, post-cleaning exploratory answers |
| **Total** | **100%** | |